<a href="https://colab.research.google.com/github/SohailVibeCoder/IB9AU---GenAI/blob/main/polish_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# Polish companies bankruptcy - fetch + first-pass audit (Google Colab)
# Paste this whole thing into one Colab cell and run it.
# ============================================================
!pip install -q ucimlrepo

from google.colab import drive
drive.mount('/content/drive')

from ucimlrepo import fetch_ucirepo
from pathlib import Path
import pandas as pd, numpy as np, json

OUT = Path("/content/drive/MyDrive/FIP_NPL_Project/polish_bankruptcy_raw")
OUT.mkdir(parents=True, exist_ok=True)

d = fetch_ucirepo(id=365)
X, y = d.data.features, d.data.targets

X.to_parquet(OUT / "X.parquet")
y.to_parquet(OUT / "y.parquet")
d.variables.to_csv(OUT / "variables.csv", index=False)
(OUT / "metadata.json").write_text(json.dumps(d.metadata, indent=2, default=str))

print("=" * 60)
print("SHAPES")
print("  X:", X.shape, "| y:", y.shape)
print("  documented: 7,027 rows for the 1st-year subset alone")

print("\nCOLUMNS")
print("  n columns:", len(X.columns))
print("  first 12:", list(X.columns)[:12])
print("  last 5:  ", list(X.columns)[-5:])
non_attr = [c for c in X.columns if not str(c).lower().startswith("attr")]
print("  NON-Attr columns:", non_attr if non_attr else "NONE")

print("\nTHE THREE STRUCTURAL QUESTIONS")
idish = [c for c in X.columns if any(k in str(c).lower() for k in ("id", "company", "firm", "name"))]
dateish = [c for c in X.columns if any(k in str(c).lower() for k in ("year", "date", "period", "time", "subset"))]
print("  1. company identifier present? ->", idish if idish else "NO")
print("  2. date / year / subset column? ->", dateish if dateish else "NO")
print("  3. duplicate feature rows (same company twice?):",
      f"{X.duplicated().sum():,} of {len(X):,}")

print("\nTARGET")
t = y.iloc[:, 0]
print("  column:", y.columns[0])
print("  counts:", t.value_counts(dropna=False).to_dict())
print(f"  positive rate: {100 * t.mean():.3f}%")

print("\nMISSINGNESS")
miss = X.isna().mean().sort_values(ascending=False)
print(f"  columns with any missing: {(miss > 0).sum()} of {len(miss)}")
print("  worst 5:")
print((100 * miss.head()).round(2).to_string())
print(f"  rows with no missing at all: {(~X.isna().any(axis=1)).sum():,} "
      f"({100 * (~X.isna().any(axis=1)).mean():.1f}%)")

print("\nSCALE / OUTLIERS (documented issue)")
desc = X.describe().T[["min", "50%", "max"]]
print("  widest ranges:")
print(desc.assign(rng=desc["max"] - desc["min"]).nlargest(5, "rng").round(1).to_string())

print("\nsaved to", OUT)
print("=" * 60)

Mounted at /content/drive
SHAPES
  X: (43405, 65) | y: (43405, 1)
  documented: 7,027 rows for the 1st-year subset alone

COLUMNS
  n columns: 65
  first 12: ['year', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11']
  last 5:   ['A60', 'A61', 'A62', 'A63', 'A64']
  NON-Attr columns: ['year', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'A33', 'A34', 'A35', 'A36', 'A37', 'A38', 'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45', 'A46', 'A47', 'A48', 'A49', 'A50', 'A51', 'A52', 'A53', 'A54', 'A55', 'A56', 'A57', 'A58', 'A59', 'A60', 'A61', 'A62', 'A63', 'A64']

THE THREE STRUCTURAL QUESTIONS
  1. company identifier present? -> NO
  2. date / year / subset column? -> ['year']
  3. duplicate feature rows (same company twice?): 401 of 43,405

TARGET
  column: class
  counts: {0: 41314, 1: 2091}
  positive ra